### Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

 ##Transformation Logic - Joining Related tables

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_pkey,
    ci.customer_id,
    ci.customer_key,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.create_date
FROM silver.crm_customers ci
LEFT JOIN silver.erp_customers ca
    ON ci.customer_key = ca.customer_number
LEFT JOIN silver.erp_customer_location la
    ON ci.customer_key = la.customer_number
"""
df = spark.sql(query)

In [0]:

df.limit(10).display()

  ###Writing to Gold table.

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")